In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import json
import uuid
import requests
from pyspark.sql import Row
import datetime

In [0]:
def ensure_metrics_table_exists():
    """
    Initializes the Gold-layer Long Format summary table if it does not exist.
    
    The table is specifically designed for incremental daily loads and backfilling:
    - Partitioned by 'report_date' to allow Atomic Partition Overwrites.
    - Includes 'updated_at' to track exactly when a metric was (re)calculated.
    - Uses Delta Lake format to support ACID transactions and Time Travel.

    Schema:
        - metric_name (String): The KPI being measured (e.g., total_compensation).
        - segment_name (String): The grouping category (business, business_type).
        - segment_value (String): The specific entity (e.g., 'Cafe Dizengoff' or 'Restaurant').
        - metric_value (Double): The aggregated monetary value.
        - report_date (Date): The logical date the data represents (Partition Key).
        - updated_at (Timestamp): Audit column for processing time.
    """
    table_name = "workspace.tel_aviv_municipality.kpi_2023_agg_compensation_payments"
    
    create_query = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        metric_name STRING,
        segment_name STRING,
        segment_value STRING,
        metric_value DOUBLE,
        report_date DATE,
        updated_at TIMESTAMP
    )
    USING DELTA
    PARTITIONED BY (report_date)
    COMMENT 'summary for daily municipal payouts for streets and business metrics.'
    """
    
    try:
        spark.sql(create_query)
        print(f"Confirmed: {table_name} infrastructure is ready.")
        return True
    except Exception as e:
        print(f"Critical Error: Table initialization failed: {str(e)}")
        return False

In [0]:
def build_metrics_summary():
    """
    Aggregates daily compensation facts into the long-format summary table.
    
    This function supports Idempotent Backfilling. By using 'INSERT OVERWRITE' 
    on a specific partition, it ensures that if a specific day is re-run, 
    the previous data for that day is replaced without affecting any other dates.


    Logic:
        1. Filters the source Fact table for only that specific day.
        2. Aggregates values by 'Business Name' and 'street name'.
        3. Overwrites the specific partition in the target Delta table.

    Returns:
        str: Status message indicating success and the date processed.
    """    

    target_table = "workspace.tel_aviv_municipality.kpi_2023_agg_compensation_payments"
    source_fact = "workspace.tel_aviv_municipality.fact_2023_daily_business_compensation"

    query = f"""
    INSERT OVERWRITE TABLE {target_table}
    
    -- Business Level Segment
    SELECT 
        'compensation_amount_ils'     AS metric_name,
        'business_name'               AS segment_name,
        business_name                 AS segment_value,
        SUM(daily_compensation_ils)   AS metric_value,
        current_date()                AS report_date,
        current_timestamp()           AS updated_at
    FROM {source_fact}
    GROUP BY business_name

    UNION ALL

    -- Category Level Segment
    SELECT 
        'compensation_amount_ils'     AS metric_name,
        'street_name'                 AS segment_name,
        street_name                   AS segment_value,
        SUM(daily_compensation_ils)   AS metric_value,
        current_date()                AS report_date,
        current_timestamp()           AS updated_at
    FROM {source_fact}
    GROUP BY street_name
    """
    
    try:
        spark.sql(query)
        return f"Successfully processed summary metrics"
    except Exception as e:
        return f"Error: Aggregation failed. Details: {str(e)}"

In [0]:
def run_metric_pipeline():
    """
    Master orchestrator to manage the end-to-end metric summary pipeline.
    
    
    Flow:
    1. Infrastructure Check: Ensures the target table exists with the right partitions.
    2. Execution: Triggers the metric summary build.
    """
    print("--- Starting metric table running ---")
    
    # Initialize
    if not ensure_metrics_table_exists():
        raise Exception("Failed to initialize target table. Pipeline aborted.")
    
    # Run Job
    status = build_metrics_summary()
    print(status)
    
    print("--- Pipeline Complete ---")

# Standard Daily Run:
run_metric_pipeline()
